# phase 3 — comparative counterfactual likelihood

this is the main experiment. we take one antecedent (the rice field is destroyed) and test 15 different consequents across 81 possible worlds. each world is identical except for population, which ranges from 30 to 270 in steps of 3.

the question we're answering: if the rice field were destroyed, what would happen? and how does the answer change depending on which worlds you look at?

lewis says: look at the closest worlds. we're going to see if that gives a different answer from looking at all worlds.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from mpl_toolkits.mplot3d import Axes3D
%matplotlib inline

plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['figure.dpi'] = 130

with open('results/results.json', 'r') as f:
    data = json.load(f)

all_results = data['all_results']
summary = data['summary']
cids = list(all_results.keys())
print(f'loaded {len(cids)} consequents across {len(all_results[cids[0]]["responses"])} worlds')

## helper functions

nothing fancy here — just computing cls at different sphere radii and variance

In [ ]:
def compute_cls_at_r(responses, r):
    valid = [resp for resp in responses if resp['answer'] in (0, 1)]
    in_sphere = [resp for resp in valid if resp['distance'] <= r]
    if not in_sphere:
        return None
    return sum(resp['answer'] for resp in in_sphere) / len(in_sphere)

def compute_variance_at_r(responses, r):
    valid = [resp for resp in responses if resp['answer'] in (0, 1)]
    in_sphere = [resp['answer'] for resp in valid if resp['distance'] <= r]
    if len(in_sphere) < 2:
        return None
    return np.var(in_sphere)

def compute_cls_curve(responses, n_bins=12):
    valid = [r for r in responses if r['answer'] in (0, 1)]
    valid.sort(key=lambda r: r['distance'])
    bin_size = max(1, len(valid) // n_bins)
    midpoints, cls_values, pops = [], [], []
    for i in range(0, len(valid), bin_size):
        chunk = valid[i:i + bin_size]
        if not chunk: continue
        midpoints.append(sum(r['distance'] for r in chunk) / len(chunk))
        pops.append(sum(r['population'] for r in chunk) / len(chunk))
        cls_values.append(sum(r['answer'] for r in chunk) / len(chunk))
    return midpoints, cls_values, pops

r_values = np.arange(0.01, 1.01, 0.0125)
print('helpers loaded')

## the ranking — what happens when the rice field is destroyed?

first let's just see the overall cls for each consequent. this is the naive view — all worlds equally weighted, no lewis sphere weighting.

In [ ]:
# compute overall cls for each consequent
ranking = []
for cid in cids:
    responses = all_results[cid]['responses']
    valid = [r for r in responses if r['answer'] in (0, 1)]
    cls = sum(r['answer'] for r in valid) / len(valid) if valid else 0
    ranking.append({'id': cid, 'name': all_results[cid]['name'], 'cls': cls})
ranking.sort(key=lambda x: x['cls'], reverse=True)

# bar chart
names = [f"{r['id']}: {r['name']}" for r in ranking]
values = [r['cls'] for r in ranking]
colors = ['#2D6A4F' if v < 0.3 else '#E9C46A' if v < 0.7 else '#E63946' for v in values]

fig, ax = plt.subplots(figsize=(14, 8))
bars = ax.barh(range(len(names)), values, color=colors, edgecolor='white', linewidth=1)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=10)
ax.set_xlabel('CLS (proportion of worlds saying YES)', fontsize=12)
ax.set_title('overall ranking — if the rice field is destroyed, what happens?', fontsize=14, fontweight='bold')
ax.set_xlim(0, 1.05)
ax.invert_yaxis()
ax.axvline(x=0.7, color='#2D6A4F', linestyle='--', alpha=0.5, label="'would' threshold (0.7)")
ax.axvline(x=0.3, color='#E63946', linestyle='--', alpha=0.5, label="'might' threshold (0.3)")
for bar, val in zip(bars, values):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.2f}', va='center', fontsize=10, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2, axis='x')
plt.tight_layout()
plt.show()

## similarity-weighted counterfactual likelihood

Lewisian weighting: closer worlds count more. $w(w_n) = 1 - d(w_n, w_0)$

In [ ]:
def weighted_cls(responses):
    valid = [r for r in responses if r['answer'] in (0, 1)]
    if not valid:
        return 0
    weights = [1 - r['distance'] for r in valid]
    num = sum(w * r['answer'] for w, r in zip(weights, valid))
    den = sum(weights)
    return num / den if den > 0 else 0

weighted_data = []
for cid in cids:
    responses = all_results[cid]['responses']
    weighted_data.append({
        'id': cid,
        'name': all_results[cid]['name'],
        'p_w': weighted_cls(responses),
    })

weighted_data.sort(key=lambda x: x['p_w'], reverse=True)

fig, ax = plt.subplots(figsize=(12, 7))

names = [f"{e['id']}: {e['name']}" for e in weighted_data]
y = np.arange(len(names))
colors = plt.cm.Blues(np.linspace(0.4, 0.85, len(names)))

ax.barh(y, [e['p_w'] for e in weighted_data], color=colors,
        edgecolor='white', linewidth=0.5, height=0.65)

ax.set_yticks(y)
ax.set_yticklabels(names, fontsize=9)
ax.set_xlabel(r'CLS$_w$(C) — similarity-weighted counterfactual likelihood', fontsize=11)
ax.set_title(
    'Counterfactual likelihood under Lewisian weighting\n'
    r'$w(w_n) = 1 - d(w_n, w_0)$',
    fontsize=13, fontweight='bold',
)
ax.set_xlim(0, 1.05)
ax.invert_yaxis()
ax.grid(True, alpha=0.15, axis='x')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print(f"\n{'Consequent':<38} {'CLS_w':>8}")
print('-' * 48)
for e in weighted_data:
    print(f"{e['id']}: {e['name']:<33} {e['p_w']:>8.3f}")

## the lewis plot — cls as the sphere expands

this is the important one. for each consequent, we compute the cls as we expand the sphere outward from the closest worlds. the left side is what lewis says matters. the right side is what the naive approach gives you.

we only plot the consequents that actually vary — the ones stuck at 0 or 1 everywhere aren't interesting

In [ ]:
colors_list = ['#E63946', '#457B9D', '#2A9D8F', '#E9C46A', '#F4A261',
               '#6C4AB6', '#1D3557', '#264653', '#D4A373', '#9B2226',
               '#BB3E03', '#005F73', '#94D2BD', '#CA6702', '#AE2012']

fig, ax = plt.subplots(figsize=(16, 9))

for i, cid in enumerate(cids):
    responses = all_results[cid]['responses']
    valid = sorted([r for r in responses if r['answer'] in (0, 1)], key=lambda x: x['distance'])
    distances, cls_vals = [], []
    cumsum = 0
    for j, resp in enumerate(valid):
        cumsum += resp['answer']
        distances.append(resp['distance'])
        cls_vals.append(cumsum / (j + 1))
    overall_cls = cls_vals[-1] if cls_vals else 0
    color = colors_list[i % len(colors_list)]
    ax.plot(distances, cls_vals, linewidth=2, color=color, alpha=0.75,
            label=f"{cid}: {all_results[cid]['name']} (CLS = {overall_cls:.2f})")

ax.axhline(y=0.5, color='#999', linestyle=':', alpha=0.4)

ax.set_xlabel('sphere radius r — normalised distance from base world W₀', fontsize=12)
ax.set_ylabel('CLS(C, S(r)) — proportion of worlds affirming consequent C', fontsize=12)
ax.set_title('cumulative counterfactual likelihood score as sphere S(r) expands',
             fontsize=13, fontweight='bold')
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.05, 1.05)
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
ax.grid(True, alpha=0.2)

ax2 = ax.twiny()
ax2.set_xlim(ax.get_xlim())
ticks = [0, 0.25, 0.5, 0.75, 1.0]
ax2.set_xticks(ticks)
ax2.set_xticklabels([f'pop {int(30 + t * 240)}' for t in ticks], fontsize=9)
ax2.set_xlabel('population at sphere boundary', fontsize=10)

plt.tight_layout()

plt.show()

## the heatmap — all consequents across all distances

each row is a consequent, each column is a population bin. the color shows how likely that outcome is in that bin. this is the full counterfactual landscape.

In [ ]:
n_bins = 10
sorted_summary = sorted(ranking, key=lambda x: x['cls'], reverse=True)
matrix = []
labels = []
x_labels = []

for s in sorted_summary:
    cid = s['id']
    responses = all_results[cid]['responses']
    valid = sorted([r for r in responses if r['answer'] in (0, 1)], key=lambda r: r['distance'])
    bin_size = max(1, len(valid) // n_bins)
    row = []
    for i in range(n_bins):
        chunk = valid[i * bin_size: (i + 1) * bin_size]
        if chunk:
            row.append(sum(r['answer'] for r in chunk) / len(chunk))
            if len(labels) == 0:
                lo = int(min(r['population'] for r in chunk))
                hi = int(max(r['population'] for r in chunk))
                x_labels.append(f'{lo}-{hi}')
    matrix.append(row)
    labels.append(f"{s['id']}: {s['name']}")

fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(matrix, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=1)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=9)
ax.set_xticks(range(len(x_labels)))
ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=8)
ax.set_xlabel('population range', fontsize=11)
ax.set_title('counterfactual heatmap — green = unlikely, red = likely', fontsize=14, fontweight='bold')
for i in range(len(matrix)):
    for j in range(len(matrix[i])):
        val = matrix[i][j]
        color = 'white' if val > 0.6 or val < 0.2 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8, color=color)
plt.colorbar(im, ax=ax, label='CLS', shrink=0.8)
plt.tight_layout()
plt.show()

## inner sphere vs outer sphere — the lewis difference

this is the money chart. blue bars are what the closest worlds say (lewis's answer). red bars are what all worlds say (naive answer). when these disagree, lewis is doing real work.

In [ ]:
inner_r = 0.125
full_r = 1.0

comparison = []
for cid in cids:
    responses = all_results[cid]['responses']
    ic = compute_cls_at_r(responses, inner_r) or 0
    fc = compute_cls_at_r(responses, full_r) or 0
    comparison.append({'id': cid, 'name': all_results[cid]['name'], 'inner': ic, 'full': fc})
comparison.sort(key=lambda x: x['inner'], reverse=True)

names = [f"{c['id']}: {c['name']}" for c in comparison]
inner_vals = [c['inner'] for c in comparison]
full_vals = [c['full'] for c in comparison]

y = np.arange(len(names))
height = 0.35

fig, ax = plt.subplots(figsize=(16, 9))
ax.barh(y - height/2, inner_vals, height, label=f'inner sphere (pop ≤ {int(30 + inner_r * 240)})', color='#1d3557', edgecolor='white', linewidth=0.5)
ax.barh(y + height/2, full_vals, height, label='all worlds (pop ≤ 270)', color='#e63946', alpha=0.6, edgecolor='white', linewidth=0.5)
ax.axvline(x=0.7, color='#f4a261', linestyle='--', linewidth=2, label='T=0.7')
ax.set_yticks(y)
ax.set_yticklabels(names, fontsize=9)
ax.set_xlabel('CLS', fontsize=12)
ax.set_title('the lewis difference\nblue = closest worlds (lewis) vs red = all worlds (naive)', fontsize=14, fontweight='bold')
ax.set_xlim(0, 1.05)
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.15, axis='x')
for i in range(len(comparison)):
    diff = full_vals[i] - inner_vals[i]
    if abs(diff) > 0.1:
        direction = '↑' if diff > 0 else '↓'
        ax.text(max(inner_vals[i], full_vals[i]) + 0.02, i, f'{direction}{abs(diff):.2f}',
                fontsize=8, va='center', color='#f4a261', fontweight='bold')
plt.tight_layout()
plt.show()

## the inner sphere detail

zooming into the 10 closest worlds. for each one, which consequents are yes and which are no? this is lewis's inner sphere laid bare — the individual world-level data.

In [ ]:
# get the 10 closest worlds
sample = sorted(all_results[cids[0]]['responses'], key=lambda r: r['distance'])[:10]
world_ids = [w['world_id'] for w in sample]
world_pops = [w['population'] for w in sample]

sorted_cids = sorted(cids, key=lambda c: compute_cls_at_r(all_results[c]['responses'], 0.125) or 0, reverse=True)
cnames = [all_results[c]['name'] for c in sorted_cids]

matrix = np.zeros((len(sorted_cids), len(world_ids)))
for ci, cid in enumerate(sorted_cids):
    responses = {r['world_id']: r['answer'] for r in all_results[cid]['responses']}
    for wi, wid in enumerate(world_ids):
        ans = responses.get(wid, -1)
        matrix[ci, wi] = ans if ans in (0, 1) else np.nan

fig, ax = plt.subplots(figsize=(14, 10))
cmap = matplotlib.colors.ListedColormap(['#2D6A4F', '#E63946'])
im = ax.imshow(matrix, cmap=cmap, aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(len(world_ids)))
ax.set_xticklabels([f'{wid}\npop={p}' for wid, p in zip(world_ids, world_pops)], fontsize=9, rotation=45, ha='right')
ax.set_yticks(range(len(sorted_cids)))
ax.set_yticklabels([f'{cid}: {name}' for cid, name in zip(sorted_cids, cnames)], fontsize=9)
ax.set_xlabel('closest worlds (inner sphere)', fontsize=12)
ax.set_title('lewis\'s inner sphere — the 10 closest worlds\nred = YES, green = NO', fontsize=13, fontweight='bold')
for ci in range(len(sorted_cids)):
    for wi in range(len(world_ids)):
        val = matrix[ci, wi]
        if not np.isnan(val):
            ax.text(wi, ci, 'Y' if val == 1 else 'N', ha='center', va='center', fontsize=10, fontweight='bold', color='white')
plt.tight_layout()
plt.show()

## co-occurrence network

which consequents tend to happen together? green edges = positive correlation (both yes in the same world). red edges = negative correlation (one yes means the other no).

In [ ]:
n = len(cids)
n_worlds = len(all_results[cids[0]]['responses'])
answer_matrix = np.zeros((n_worlds, n))
for j, cid in enumerate(cids):
    for w, r in enumerate(all_results[cid]['responses']):
        answer_matrix[w, j] = r['answer'] if r['answer'] in (0, 1) else np.nan

valid_mask = ~np.isnan(answer_matrix).any(axis=1)
clean = answer_matrix[valid_mask]
corr = np.corrcoef(clean.T)

fig, ax = plt.subplots(figsize=(14, 14))
angles = np.linspace(0, 2 * np.pi, n, endpoint=False)
radius = 4
positions = {cids[i]: (radius * np.cos(a), radius * np.sin(a)) for i, a in enumerate(angles)}
names_dict = {cid: all_results[cid]['name'] for cid in cids}
cls_dict = {r['id']: r['cls'] for r in ranking}

for i in range(n):
    for j in range(i + 1, n):
        c = corr[i, j]
        if abs(c) > 0.15 and not np.isnan(c):
            x = [positions[cids[i]][0], positions[cids[j]][0]]
            y = [positions[cids[i]][1], positions[cids[j]][1]]
            color = '#2D6A4F' if c > 0 else '#E63946'
            ax.plot(x, y, color=color, alpha=min(abs(c) * 1.5, 0.9), linewidth=abs(c) * 4, zorder=1)

for cid in cids:
    x, y = positions[cid]
    cls_val = cls_dict.get(cid, 0)
    nc = '#E63946' if cls_val >= 0.7 else '#E9C46A' if cls_val >= 0.3 else '#2D6A4F'
    circle = plt.Circle((x, y), 0.4, color=nc, zorder=3, ec='white', linewidth=2)
    ax.add_patch(circle)
    lx, ly = x * 1.35, y * 1.35
    ha = 'left' if x > 0 else 'right' if x < 0 else 'center'
    ax.annotate(f"{cid}\n{names_dict[cid]}\n({cls_val:.2f})", (x, y), (lx, ly),
                fontsize=8, ha=ha, va='center', fontweight='bold',
                arrowprops=dict(arrowstyle='-', color='#999', lw=0.5))

ax.set_xlim(-7, 7); ax.set_ylim(-7, 7); ax.set_aspect('equal'); ax.axis('off')
ax.set_title('co-occurrence network\ngreen = happen together, red = one blocks the other', fontsize=13, fontweight='bold')
ax.plot([], [], color='#2D6A4F', linewidth=3, label='positive correlation')
ax.plot([], [], color='#E63946', linewidth=3, label='negative correlation')
ax.legend(loc='lower right', fontsize=10)
plt.tight_layout()
plt.show()

## 3d surface — the counterfactual landscape

x = population, y = consequent, z = cls. the full terrain in one view.

In [ ]:
n_bins = 12
sorted_by_cls = sorted(cids, key=lambda c: compute_cls_at_r(all_results[c]['responses'], 1.0) or 0, reverse=True)

Z = np.zeros((len(sorted_by_cls), n_bins))
X_labels = []
for ci, cid in enumerate(sorted_by_cls):
    responses = sorted([r for r in all_results[cid]['responses'] if r['answer'] in (0,1)], key=lambda r: r['distance'])
    bs = max(1, len(responses) // n_bins)
    for bi in range(n_bins):
        chunk = responses[bi*bs:(bi+1)*bs]
        if chunk:
            Z[ci, bi] = sum(r['answer'] for r in chunk) / len(chunk)
            if ci == 0:
                X_labels.append(str(int(sum(r['population'] for r in chunk)/len(chunk))))

X, Y = np.meshgrid(range(n_bins), range(len(sorted_by_cls)))
fig = plt.figure(figsize=(18, 12))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(X, Y, Z, cmap='RdYlGn_r', alpha=0.85, edgecolor='white', linewidth=0.3)
ax.set_xlabel('\npopulation →', fontsize=11, labelpad=15)
ax.set_ylabel('\nconsequent', fontsize=11, labelpad=15)
ax.set_zlabel('\nCLS', fontsize=11, labelpad=10)
ax.set_title('3d counterfactual landscape', fontsize=14, fontweight='bold', pad=20)
ax.set_xticks(range(min(n_bins, len(X_labels))))
ax.set_xticklabels(X_labels[:n_bins], fontsize=7, rotation=45)
ax.set_yticks(range(len(sorted_by_cls)))
ax.set_yticklabels(sorted_by_cls, fontsize=7)
ax.set_zlim(0, 1)
ax.view_init(elev=25, azim=135)
fig.colorbar(surf, ax=ax, shrink=0.5, label='CLS')
fig.subplots_adjust(left=0.05, right=0.95, bottom=0.05, top=0.95)
plt.show()

In [ ]:
# why c1, c11, and c12 are interesting
# each one shows a different pattern in lewis's inner sphere

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

targets = [
    ('C1', 'violent conflict', '#e63946'),
    ('C11', 'rationing established', '#2a9d8f'),
    ('C12', 'someone dies of starvation', '#e9c46a'),
]

for idx, (cid, name, color) in enumerate(targets):
    ax = axes[idx]
    responses = all_results[cid]['responses']
    valid = sorted([r for r in responses if r['answer'] in (0, 1)], key=lambda r: r['distance'])
    
    # cumulative cls
    cum_yes, cum_total = 0, 0
    cum_d, cum_cls = [], []
    for r in valid:
        cum_total += 1
        cum_yes += r['answer']
        cum_d.append(r['distance'])
        cum_cls.append(cum_yes / cum_total)
    
    ax.plot(cum_d, cum_cls, color=color, linewidth=3)
    ax.fill_between(cum_d, cum_cls, alpha=0.15, color=color)
    
    # inner sphere line
    ax.axvline(x=0.125, color='#f4a261', linestyle='--', linewidth=2, alpha=0.8)
    ax.axhline(y=0.5, color='#999', linestyle=':', alpha=0.5)
    
    # scatter the raw answers along the bottom
    for r in valid:
        c = '#e63946' if r['answer'] == 1 else '#2a9d8f'
        ax.scatter(r['distance'], -0.03, color=c, s=8, alpha=0.6)
    
    # annotate
    inner_count = sum(1 for r in valid if r['distance'] <= 0.125)
    inner_cls = cum_cls[inner_count - 1] if inner_count > 0 else 0
    full_cls = cum_cls[-1]
    ax.text(0.02, 0.95, f'inner sphere: {inner_cls:.2f}', transform=ax.transAxes,
            fontsize=10, fontweight='bold', color='#f4a261', va='top')
    ax.text(0.02, 0.87, f'all worlds: {full_cls:.2f}', transform=ax.transAxes,
            fontsize=10, fontweight='bold', color=color, va='top')
    
    ax.set_title(f'{cid}: {name}', fontsize=13, fontweight='bold')
    ax.set_xlabel('sphere radius r')
    ax.set_ylabel('CLS' if idx == 0 else '')
    ax.set_xlim(-0.02, 1.05)
    ax.set_ylim(-0.08, 1.05)
    ax.grid(True, alpha=0.2)

plt.suptitle('three patterns in lewis\'s inner sphere\n'
             'orange dots at bottom = raw YES/NO per world, curve = cumulative CLS',
             fontsize=14, fontweight='bold', y=1.04)
plt.tight_layout()
plt.show()

## c12 deep dive — starvation

c12 is the most interesting consequent because the closest worlds are genuinely split. this is lewis's indeterminacy in action.

In [ ]:
responses = all_results['C12']['responses']
valid = sorted([r for r in responses if r['answer'] in (0, 1)], key=lambda r: r['distance'])

# cumulative cls
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# plot 1: cumulative cls
ax = axes[0, 0]
cum_yes, cum_total = 0, 0
cum_d, cum_cls = [], []
for r in valid:
    cum_total += 1; cum_yes += r['answer']
    cum_d.append(r['distance']); cum_cls.append(cum_yes / cum_total)
ax.plot(cum_d, cum_cls, color='#e63946', linewidth=3)
ax.fill_between(cum_d, cum_cls, alpha=0.15, color='#e63946')
ax.axvline(x=0.125, color='#f4a261', linestyle='--', linewidth=2, alpha=0.8)
ax.axhline(y=0.5, color='#999', linestyle=':', alpha=0.5)
ax.set_title('c12: cumulative cls as sphere expands', fontweight='bold')
ax.set_xlabel('sphere radius r'); ax.set_ylabel('CLS')
ax.set_xlim(-0.02, 1.05); ax.set_ylim(-0.05, 1.05)
ax.grid(True, alpha=0.2)

# plot 2: individual responses
ax = axes[0, 1]
pops = [r['population'] for r in valid]
ans = [r['answer'] for r in valid]
colors = ['#2a9d8f' if a == 0 else '#e63946' for a in ans]
y_jitter = [0.7 if a == 1 else 0.3 for a in ans]
ax.scatter(pops, y_jitter, c=colors, s=60, alpha=0.7, edgecolors='white', linewidth=0.5)
ax.axvspan(30, 30 + 0.125 * 240, alpha=0.08, color='#f4a261')
ax.set_title('c12: individual responses', fontweight='bold')
ax.set_xlabel('population'); ax.set_yticks([0.3, 0.7]); ax.set_yticklabels(['NO', 'YES'])
ax.set_xlim(25, 275)

# plot 3: sliding window
ax = axes[1, 0]
window = 8
mid_d, local = [], []
for i in range(len(valid) - window + 1):
    chunk = valid[i:i+window]
    mid_d.append((chunk[0]['distance'] + chunk[-1]['distance']) / 2)
    local.append(sum(r['answer'] for r in chunk) / len(chunk))
ax.plot(mid_d, local, color='#e63946', linewidth=2.5)
ax.fill_between(mid_d, local, alpha=0.15, color='#e63946')
ax.axvline(x=0.125, color='#f4a261', linestyle='--', linewidth=2, alpha=0.8)
ax.axhline(y=0.5, color='#999', linestyle=':', alpha=0.5)
ax.set_title('c12: sliding window (local starvation rate)', fontweight='bold')
ax.set_xlabel('distance'); ax.set_ylabel('local CLS')
ax.set_xlim(-0.02, 1.05); ax.set_ylim(-0.05, 1.05)
ax.grid(True, alpha=0.2)

# plot 4: lewis verdict
ax = axes[1, 1]
inner = valid[:10]; mid = valid[20:40]; outer = valid[-10:]
vals = [sum(r['answer'] for r in s) / len(s) for s in [inner, mid, outer, valid]]
labels = ['inner\n(30-57)', 'middle\n(90-147)', 'outer\n(240-270)', 'all\n(30-270)']
bar_colors = ['#1d3557', '#457b9d', '#e63946', '#888']
bars = ax.bar(range(4), vals, color=bar_colors, width=0.6, edgecolor='white', linewidth=2)
ax.axhline(y=0.8, color='#f4a261', linestyle='--', linewidth=2, alpha=0.6)
ax.axhline(y=0.5, color='#999', linestyle=':', alpha=0.5)
for bar, val in zip(bars, vals):
    v = 'WOULD' if val >= 0.8 else 'MIGHT' if val > 0 else 'WOULD NOT'
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.03, f'CLS={val:.2f}\n{v}',
            ha='center', fontsize=10, fontweight='bold')
ax.set_xticks(range(4)); ax.set_xticklabels(labels)
ax.set_title('c12: lewis verdict vs naive verdict', fontweight='bold')
ax.set_ylim(0, 1.15)
ax.grid(True, alpha=0.15, axis='y')

plt.suptitle('c12: someone dies of starvation — deep dive', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## variance analysis — the key finding

here's the interesting part. lewis says the closest worlds give you the answer. but our data shows the closest worlds are where the MOST disagreement is. the variance is highest at the inner sphere.

we classify each consequent into three types:
- **robust**: same answer everywhere, low variance at all distances
- **distance-sensitive**: verdict changes depending on sphere size
- **indeterminate**: closest worlds disagree — no clear truth value

In [ ]:
cat_colors = {'ROBUST': '#2a9d8f', 'DISTANCE-SENSITIVE': '#e9c46a', 'INDETERMINATE': '#e63946'}

# compute stats for each consequent
stats = {}
for cid in cids:
    responses = all_results[cid]['responses']
    inner_var = compute_variance_at_r(responses, 0.125) or 0
    outer_var = compute_variance_at_r(responses, 1.0) or 0
    inner_cls = compute_cls_at_r(responses, 0.125) or 0
    overall_cls = compute_cls_at_r(responses, 1.0) or 0
    
    # classify
    if inner_var < 0.05 and outer_var < 0.05:
        cat = 'ROBUST'
    elif inner_var > 0.15:
        cat = 'INDETERMINATE'
    elif abs(overall_cls - inner_cls) > 0.2:
        cat = 'DISTANCE-SENSITIVE'
    elif inner_var > 0.05:
        cat = 'INDETERMINATE'
    else:
        cat = 'ROBUST'
    
    stats[cid] = {
        'name': all_results[cid]['name'], 'category': cat,
        'inner_var': inner_var, 'outer_var': outer_var,
        'inner_cls': inner_cls, 'overall_cls': overall_cls,
    }

# print classification
for cat in ['ROBUST', 'DISTANCE-SENSITIVE', 'INDETERMINATE']:
    print(f'\n{cat}:')
    for cid, s in stats.items():
        if s['category'] == cat:
            print(f"  {cid}: {s['name']:<28} inner_var={s['inner_var']:.3f} inner_cls={s['inner_cls']:.2f} overall_cls={s['overall_cls']:.2f}")

In [ ]:
# variance curves for all consequents
fig, ax = plt.subplots(figsize=(16, 8))
for cid in cids:
    variances = [compute_variance_at_r(all_results[cid]['responses'], r) or 0 for r in r_values]
    cat = stats[cid]['category']
    ax.plot(r_values, variances, linewidth=2, color=cat_colors[cat], alpha=0.7,
            label=f"{cid}: {stats[cid]['name']} [{cat}]")

ax.axvline(x=0.125, color='#f4a261', linestyle='--', linewidth=2, alpha=0.6, label='inner sphere')
ax.set_xlabel('sphere radius r', fontsize=12)
ax.set_ylabel('variance of answers within sphere', fontsize=12)
ax.set_title('variance across spheres — how much do worlds disagree?\nhigh variance = uncertain, low = robust', fontsize=14, fontweight='bold')
ax.set_xlim(-0.02, 1.05); ax.set_ylim(-0.01, 0.30)
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# classification scatter
fig, ax = plt.subplots(figsize=(12, 10))
for cid in cids:
    s = stats[cid]
    gradient = abs(s['overall_cls'] - s['inner_cls'])
    ax.scatter(s['inner_var'], gradient, s=200, color=cat_colors[s['category']],
               edgecolors='white', linewidth=1.5, zorder=3, alpha=0.85)
    ax.annotate(f"{cid}: {s['name']}", (s['inner_var'], gradient),
                textcoords='offset points', xytext=(8, 4), fontsize=8, color='#ccc')

ax.axhline(y=0.2, color='#e9c46a', linestyle=':', alpha=0.4)
ax.axvline(x=0.15, color='#e63946', linestyle=':', alpha=0.4)
ax.text(0.01, 0.02, 'ROBUST', fontsize=12, color='#2a9d8f', fontweight='bold', alpha=0.7)
ax.text(0.01, 0.35, 'DISTANCE-SENSITIVE', fontsize=12, color='#e9c46a', fontweight='bold', alpha=0.7)
ax.text(0.18, 0.02, 'INDETERMINATE', fontsize=12, color='#e63946', fontweight='bold', alpha=0.7)
ax.set_xlabel('variance at inner sphere', fontsize=12)
ax.set_ylabel('|CLS(all) − CLS(inner)|', fontsize=12)
ax.set_title('classifying counterfactuals by variance profile', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.15)
for cat, color in cat_colors.items():
    ax.scatter([], [], s=100, color=color, label=cat, edgecolors='white')
ax.legend(loc='upper right', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# inner vs outer variance
fig, ax = plt.subplots(figsize=(10, 10))
for cid in cids:
    s = stats[cid]
    ax.scatter(s['inner_var'], s['outer_var'], s=180, color=cat_colors[s['category']],
               edgecolors='white', linewidth=1.5, zorder=3)
    ax.annotate(cid, (s['inner_var'], s['outer_var']), textcoords='offset points',
                xytext=(6, 4), fontsize=9, color='#ccc', fontweight='bold')

lim = max(ax.get_xlim()[1], ax.get_ylim()[1]) + 0.02
ax.plot([0, lim], [0, lim], color='#666', linestyle=':', alpha=0.5)
ax.text(lim*0.7, lim*0.5, 'variance DECREASES\nwith distance', fontsize=9, color='#2a9d8f', ha='center', rotation=45, alpha=0.7)
ax.text(lim*0.3, lim*0.6, 'variance INCREASES\nwith distance', fontsize=9, color='#e63946', ha='center', rotation=45, alpha=0.7)
ax.set_xlabel('variance at inner sphere', fontsize=12)
ax.set_ylabel('variance at full sphere', fontsize=12)
ax.set_title('does uncertainty decrease with distance?', fontsize=14, fontweight='bold')
ax.set_aspect('equal'); ax.grid(True, alpha=0.15)
for cat, color in cat_colors.items():
    ax.scatter([], [], s=100, color=color, label=cat, edgecolors='white')
ax.legend(loc='upper left', fontsize=10)
plt.tight_layout()
plt.show()

## verdict map — how verdicts change as the sphere expands

this is the core lewis insight. fix a threshold (say 0.8). then sweep the sphere radius from small to large. at each radius, color each consequent red if it qualifies as "would" and dark if it doesn't. you can literally see consequents flipping as you include more distant worlds.

In [ ]:
threshold = 0.8
r_sweep = np.arange(0.01, 1.01, 0.0125)

sorted_cids_inner = sorted(cids, key=lambda c: compute_cls_at_r(all_results[c]['responses'], 0.125) or 0, reverse=True)
names_sorted = [f"{cid}: {all_results[cid]['name']}" for cid in sorted_cids_inner]

cls_matrix = np.zeros((len(sorted_cids_inner), len(r_sweep)))
for ci, cid in enumerate(sorted_cids_inner):
    responses = all_results[cid]['responses']
    for ri, r in enumerate(r_sweep):
        cls = compute_cls_at_r(responses, r)
        cls_matrix[ci, ri] = cls if cls is not None else 0

verdict_matrix = (cls_matrix >= threshold).astype(float)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 16))

# binary verdict map
cmap = matplotlib.colors.ListedColormap(['#1a2a1a', '#e63946'])
ax1.imshow(verdict_matrix, cmap=cmap, aspect='auto', vmin=0, vmax=1, interpolation='nearest')
n_ticks = 10
tick_pos = np.linspace(0, len(r_sweep)-1, n_ticks).astype(int)
ax1.set_xticks(tick_pos)
ax1.set_xticklabels([f'r={r_sweep[i]:.2f}\npop≤{int(30+r_sweep[i]*240)}' for i in tick_pos], fontsize=8)
ax1.set_yticks(range(len(names_sorted))); ax1.set_yticklabels(names_sorted, fontsize=9)
ax1.set_title(f'verdict map (T={threshold}) — red = WOULD, dark = would not/might', fontsize=14, fontweight='bold')
inner_idx = np.argmin(np.abs(r_sweep - 0.125))
ax1.axvline(x=inner_idx, color='#f4a261', linestyle='--', linewidth=2, alpha=0.8)

# continuous cls heatmap
im = ax2.imshow(cls_matrix, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=1, interpolation='bilinear')
ax2.set_xticks(tick_pos)
ax2.set_xticklabels([f'r={r_sweep[i]:.2f}\npop≤{int(30+r_sweep[i]*240)}' for i in tick_pos], fontsize=8)
ax2.set_yticks(range(len(names_sorted))); ax2.set_yticklabels(names_sorted, fontsize=9)
ax2.set_title('continuous cls heatmap', fontsize=14, fontweight='bold')
ax2.axvline(x=inner_idx, color='#f4a261', linestyle='--', linewidth=2, alpha=0.8)
plt.colorbar(im, ax=ax2, label='CLS', shrink=0.8)

plt.tight_layout()
plt.show()

## the key finding

lewis's inner sphere is not a zone of clarity — it's a zone of maximum uncertainty. the closest worlds disagree more than the full set of worlds. the threshold T is not optional scaffolding on lewis's theory, it's load-bearing. without it, the inner sphere gives you a number but no verdict.

this suggests that counterfactuals about small perturbations to reality are genuinely harder to evaluate than counterfactuals about large perturbations. change one small thing and the outcome is anyone's guess. change everything dramatically and the outcome becomes obvious. lewis was right that the closest worlds matter most — but "mattering most" doesn't mean "giving the clearest answer." it means giving the most RELEVANT answer, even when that answer is: we don't know.

In [ ]:
# final summary: average variance at inner vs outer
inner_vars = [stats[cid]['inner_var'] for cid in cids]
outer_vars = [stats[cid]['outer_var'] for cid in cids]

print(f'average variance at inner sphere: {np.mean(inner_vars):.4f}')
print(f'average variance at full sphere:  {np.mean(outer_vars):.4f}')
print()
if np.mean(inner_vars) > np.mean(outer_vars):
    print('→ the closest worlds show MORE disagreement than all worlds combined.')
    print('  lewis\'s inner sphere is a zone of maximum uncertainty.')
else:
    print('→ variance is similar or higher at the outer sphere.')

In [ ]:
comparison = []
for cid in cids:
    responses = all_results[cid]['responses']
    inner_cls = compute_cls_at_r(responses, 0.125) or 0
    full_cls = compute_cls_at_r(responses, 1.0) or 0
    comparison.append({
        'id': cid, 'name': all_results[cid]['name'],
        'inner_cls': inner_cls, 'full_cls': full_cls,
        'diff': abs(full_cls - inner_cls),
    })
comparison.sort(key=lambda x: x['diff'], reverse=True)

fig, ax = plt.subplots(figsize=(12, 8))
names = [f"{c['id']}: {c['name']}" for c in comparison]
diffs = [c['diff'] for c in comparison]
bar_colors = ['#e63946' if d > 0.15 else '#e9c46a' if d > 0.05 else '#2a2a30' for d in diffs]

ax.barh(range(len(names)), diffs, color=bar_colors, edgecolor='white', linewidth=0.5)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=9)
ax.set_xlabel('|CLS(S(1.0)) − CLS(S(0.125))|', fontsize=11)
ax.set_title('sensitivity of counterfactual verdict to sphere radius\n'
             'S(0.125): 10 closest worlds, pop 30–57  |  S(1.0): all 81 worlds, pop 30–270',
             fontsize=12, fontweight='bold')
ax.invert_yaxis()
ax.grid(True, alpha=0.15, axis='x')
plt.tight_layout()
plt.show()